In [2]:
import cv2
import os
import numpy as np 
import skimage.filters
import skimage.color
import skimage.io
import matplotlib.pyplot as plt
import shapely
from shapely.geometry.polygon import LinearRing, Polygon, LineString, Point
from shapely.geometry import MultiPolygon
import pandas as pd
from area import area
from shapely.ops import cascaded_union
from PIL import Image
from keras.layers import *

cwd = os.getcwd()
#os.environ["PATH"] += os.pathsep + "C:/LAB/supporting/DOT"
#os.environ["PATH"] += os.pathsep + cwd + "\\DOT"

In [3]:
from tensorflow.keras.models import load_model
model_color = load_model('building-detector-v11-colored.h5')
model_gray = load_model('building-detector-v12-gray.h5')

In [16]:
#"images/done/7.png"
#2957,2762,2351,2398,2360,2949,2970,703
#,3054,3081,3354
image_path="E:/IMAGEPRO/geotiff/tiffs/tiffs.2762.tif"
#input:image_path
#input : chunk_size=50
#input : img_size = 224

width=500
height=500
chunk_size=50
y_size=chunk_size 
x_size=chunk_size 

img_size = 224
img = cv2.imread(image_path)
imS = cv2.resize(img, (500,500)) 

#####################
imagelist_c=[]
imagelist_g=[]
all_grids=[]
building_grids=[]
other_grids=[]
idx=0
for ys in np.arange(0,height,y_size):
    for xs in np.arange(0,width,x_size):
        if(ys<=height-y_size and xs<=width-x_size):
            #print(ys,xs)
            idx=idx+1
            start_point = (ys,xs)
            end_point = (ys+y_size,xs+x_size)
            
            all_grids.append((xs,ys))
            
            crop_img = imS[ys:ys+y_size, xs:xs+x_size][...,::-1] 
            
            
            resized_arr = cv2.resize(crop_img, (img_size, img_size))
            imagelist_c.append(resized_arr)

            img_gray = cv2.cvtColor(crop_img, cv2.COLOR_BGR2GRAY)
            ret, thresh = cv2.threshold(img_gray, 125, 255, cv2.THRESH_BINARY)
            resized_arr2 = cv2.resize(thresh, (img_size, img_size)) # Reshaping images to preferred size
            imagelist_g.append(resized_arr2)
                
                
img_size = 224
imagelist_in_c = np.array(imagelist_c) / 255
imagelist_in_c.reshape(-1, img_size, img_size, 1)

imagelist_in_g = np.array(imagelist_g) / 255
imagelist_in_g.reshape(-1, img_size, img_size, 1)

predict_x_c = model_color.predict(imagelist_in_c)
predict_x_g = model_gray.predict(imagelist_in_g)




classes_x_c=np.argmax(predict_x_c,axis=1)
classes_x_g=np.argmax(predict_x_g,axis=1)


#ARGNAX
for coords,cls,pred,cls_g,pred_g in zip(all_grids,classes_x_c,predict_x_c,classes_x_g,predict_x_g):
    if(pred[0]>=0.9 and cls_g==0): # and abs(pred[0]-pred[1])>=0.1
        print(pred[0],pred_g[0])
        xs_=coords[0]
        ys_=coords[1]
        cv2.rectangle(imS, (xs_,ys_), (xs_+x_size,ys_+y_size), (0, 0, 255), 2)
"""
#THESH
for coords,cls in zip(all_grids,predict_x):
    print(cls)
    if(cls[0]==0):
        #print(cls)
        xs_=coords[0]
        ys_=coords[1]
        cv2.rectangle(imS, (xs_,ys_), (xs_+x_size,ys_+y_size), (0, 0, 255), 2)        
"""        
cv2.namedWindow("Window")

# Starting The Loop So Image Can Be Shown

wait_time = 1000
while cv2.getWindowProperty('Window', cv2.WND_PROP_VISIBLE) >= 1:
    cv2.imshow("Window",imS)
    keyCode = cv2.waitKey(wait_time)
    if (keyCode & 0xFF) == ord("q"):
        cv2.destroyAllWindows()
        break

4/4 [==============================] - 0s 99ms/step
0.99928015 0.6830256
0.99866056 0.837681
0.9912868 0.67516905
0.9957099 0.8569386
0.99971205 0.6076948
0.9993111 0.75268614
0.9980902 0.69501084
0.9965365 0.6579365
0.9578438 0.7556858
0.99850523 0.92184675
0.99611074 0.655317
0.95292455 0.72666603
0.9813151 0.61859673
0.9876916 0.5497746
0.9995728 0.87985915
0.9615468 0.68486404
0.9958605 0.5362683
0.996837 0.65852815
0.9977749 0.8981859
0.9999169 0.64374214
0.9662923 0.6817142
0.9752357 0.8781992
